# EcoWatt GPU service (your `advice_engine.py`, unchanged)
Replaces the vLLM notebook. Runs the **same engine you evaluated** (Qwen3-4B-Instruct-2507, 4-bit NF4, LM Format Enforcer, greedy decoding)
behind a small HTTP service, exposed with ngrok.

**Before running**
1. Add Kaggle Secrets: `GPU_API_KEY` (long random string; also set as `LLM_API_KEY` in the backend), `NGROK_AUTH_TOKEN`,
   optionally `BACKEND_URL` and `BACKEND_ADMIN_TOKEN` (to register the tunnel URL automatically; skip them while the backend runs on localhost).
2. Upload this project's `gpu_service/` folder and the 4 hand-off files (`advice_catalog.json`, `ecowatt_system_prompt.txt`,
   `ecowatt_safety_rules.json`, `ecowatt_required_prefixes.json`) as a Kaggle Dataset (`./scripts/build_kaggle_zip.sh` builds `dist/ecowatt-gpu.zip` with both); the next cells find it automatically.
3. Run all cells in order, in every new session (the install cell too). No restart needed.
4. If an ngrok token was ever pasted into an older notebook, revoke it in the ngrok dashboard.


In [ ]:
%pip install -q "transformers>=5.0.0,<6.0.0" accelerate bitsandbytes fastapi "uvicorn[standard]" pydantic pyngrok httpx python-multipart easyocr
%pip install -q git+https://github.com/noamgat/lm-format-enforcer
# Run this in EVERY new Kaggle session (each session is a fresh machine). No restart is needed afterwards:
# the GPU service runs as a separate process and picks up the new packages.

In [ ]:
import os, subprocess, sys, time, threading, httpx

_missing = subprocess.run([sys.executable, "-c", "import lmformatenforcer, bitsandbytes, transformers"],
                          capture_output=True, text=True)
if _missing.returncode != 0:
    raise RuntimeError("Packages missing - run the install cell above first (needed in every new session).\n"
                       + _missing.stderr.strip().splitlines()[-1])
from kaggle_secrets import UserSecretsClient

sec = UserSecretsClient()
import glob
# Find the uploaded dataset wherever Kaggle mounted it (path differs between Kaggle versions)
_server = sorted(glob.glob("/kaggle/input/**/gpu_service/server.py", recursive=True))
_catalog = sorted(glob.glob("/kaggle/input/**/handoff/advice_catalog.json", recursive=True))
if not _server or not _catalog:
    raise RuntimeError("Upload ecowatt-gpu.zip as a Kaggle Dataset and add it to this notebook (Add Input).")
PROJECT_DIR = os.path.dirname(os.path.dirname(_server[0]))   # folder that CONTAINS gpu_service/
HANDOFF_DIR = os.path.dirname(_catalog[0])                     # folder with the 4 hand-off files
print("PROJECT_DIR:", PROJECT_DIR, "| HANDOFF_DIR:", HANDOFF_DIR)

env = dict(os.environ,
           GPU_API_KEY=sec.get_secret("GPU_API_KEY"),
           HANDOFF_DIR=HANDOFF_DIR,
           HF_HOME="/kaggle/working/hf_cache",
           PYTHONPATH=PROJECT_DIR)

proc = subprocess.Popen([sys.executable, "-m", "uvicorn", "gpu_service.server:app",
                         "--host", "0.0.0.0", "--port", "8000", "--workers", "1"],   # exactly ONE worker
                        cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
threading.Thread(target=lambda: [print(l, end="") for l in proc.stdout], daemon=True).start()

for i in range(90):                                    # model download + load can take several minutes
    time.sleep(10)
    if proc.poll() is not None:
        raise RuntimeError(f"GPU service exited with code {proc.returncode} - see the log above.")
    try:
        r = httpx.get("http://127.0.0.1:8000/health", timeout=3)
        if r.status_code == 200:
            print("\nREADY:", r.json()); break
    except Exception:
        print(".", end="", flush=True)
else:
    raise RuntimeError("GPU service did not become healthy within 15 minutes.")

In [ ]:
from pyngrok import conf, ngrok
# Pass the token in the config instead of ngrok.set_auth_token(): on a re-run the token is already saved,
# newer ngrok agents answer "Authtoken already set" and pyngrok treats that as an error.
conf.get_default().auth_token = sec.get_secret("NGROK_AUTH_TOKEN")
try: ngrok.kill()
except Exception: pass
public_url = ngrok.connect("http://127.0.0.1:8000").public_url
print("GPU endpoint:", public_url)

# Register the new URL with the backend so nobody has to copy it by hand.
try:
    backend, token = sec.get_secret("BACKEND_URL"), sec.get_secret("BACKEND_ADMIN_TOKEN")
    r = httpx.put(backend.rstrip("/") + "/admin/llm-endpoint", json={"url": public_url},
                  headers={"X-Admin-Token": token}, timeout=20)
    print("Backend registration:", r.status_code, r.text[:200])
except Exception as e:
    print("Not registered automatically (", type(e).__name__, "). Set LLM_BASE_URL in the backend to the URL above.")